# Self-Correcting RAG Agent with LangGraph and OpenVINO

Standard Retrieval-Augmented Generation (RAG) pipelines follow a linear path: retrieve documents, then generate an answer. If the retrieval step returns irrelevant documents, the model often hallucinates.

This notebook demonstrates a **self-correcting RAG agent** that uses [LangGraph](https://langchain-ai.github.io/langgraph/) to orchestrate a stateful workflow with conditional branching. The agent:

1. **Retrieves** documents from a vector store
2. **Grades** each document for relevance using an LLM
3. **Generates** an answer if documents are relevant
4. **Rewrites** the query and re-retrieves if documents are irrelevant

All LLM inference runs on [OpenVINO](https://docs.openvino.ai/) for optimized performance on Intel hardware.

![self-correcting-rag](https://raw.githubusercontent.com/langchain-ai/langgraph/main/docs/docs/tutorials/rag/img/langgraph_self_rag.png)


#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Prepare Model and Tokenizer](#Prepare-Model-and-Tokenizer)
  - [Select Model](#Select-Model)
  - [Convert and Compress Model](#Convert-and-Compress-Model)
  - [Select Inference Device](#Select-Inference-Device)
  - [Load Model with OpenVINO](#Load-Model-with-OpenVINO)
- [Build the Knowledge Base](#Build-the-Knowledge-Base)
  - [Load and Chunk Documents](#Load-and-Chunk-Documents)
  - [Create Embeddings and Vector Store](#Create-Embeddings-and-Vector-Store)
- [Define the RAG Agent Graph](#Define-the-RAG-Agent-Graph)
  - [Define Agent State](#Define-Agent-State)
  - [Retrieval Node](#Retrieval-Node)
  - [Document Grader Node](#Document-Grader-Node)
  - [Answer Generator Node](#Answer-Generator-Node)
  - [Query Rewriter Node](#Query-Rewriter-Node)
  - [Routing Logic](#Routing-Logic)
  - [Assemble the Graph](#Assemble-the-Graph)
- [Run the Agent](#Run-the-Agent)
- [Interactive Demo with Gradio](#Interactive-Demo-with-Gradio)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/llm-agent-langgraph/llm-agent-langgraph.ipynb" />

## Prerequisites

[back to top ⬆️](#Table-of-contents:)

Install required dependencies.

In [ ]:
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w", encoding="utf-8").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("llm-agent-langgraph.ipynb")

In [ ]:
import os

os.environ["GIT_CLONE_PROTECTION_ACTIVE"] = "false"

%pip install -Uq pip
%pip uninstall -q -y optimum optimum-intel optimum-onnx
%pip install --pre -Uq "openvino>=2025.3.0" openvino-tokenizers[transformers] --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly
%pip install -q --extra-index-url https://download.pytorch.org/whl/cpu "transformers==4.53.3" \
    "langchain>=0.3.0,<1.0" \
    "langchain-community>=0.3.0,<1.0" \
    "langchain-huggingface>=0.1.2" \
    "langgraph>=0.2.0,<1.0" \
    "chromadb>=0.5.0,<1.0" \
    "sentence-transformers>=3.0.0" \
    "nncf>=2.18.0" \
    "torch==2.8" \
    "torchvision==0.23.0" \
    "datasets<4.0.0" \
    "accelerate" \
    "bs4" \
    "gradio>=4.19,<6" \
    "ipywidgets" \
    "huggingface-hub>=0.26.5"
%pip install -q "git+https://github.com/huggingface/optimum-intel.git" --extra-index-url https://download.pytorch.org/whl/cpu

## Prepare Model and Tokenizer

[back to top ⬆️](#Table-of-contents:)

We use [Phi-3-mini-4k-instruct](https://huggingface.co/microsoft/Phi-3-mini-4k-instruct) as our LLM. It is small enough for fast inference yet capable enough for document grading and answer generation. The model is converted to OpenVINO IR format and compressed to INT4 for efficient inference.

### Select Model

[back to top ⬆️](#Table-of-contents:)

In [ ]:
import ipywidgets as widgets

model_ids = [
    "microsoft/Phi-3-mini-4k-instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "meta-llama/Llama-3.2-1B-Instruct",
]

llm_model_id = widgets.Dropdown(
    options=model_ids,
    value=model_ids[0],
    description="Model:",
    disabled=False,
)

llm_model_id

### Convert and Compress Model

[back to top ⬆️](#Table-of-contents:)

Export the model to OpenVINO IR format with INT4 weight compression using the Optimum CLI. This reduces memory footprint and speeds up inference significantly.

In [ ]:
import gc
from cmd_helper import optimum_cli

llm_model_path = llm_model_id.value.split("/")[-1]

model_path = Path(llm_model_path) / "INT4"

if not model_path.exists():
    optimum_cli(
        llm_model_id.value,
        model_path,
        additional_args={
            "task": "text-generation-with-past",
            "weight-format": "int4",
            "group-size": "128",
            "ratio": "1.0",
            "trust-remote-code": "",
        },
    )
    gc.collect()
else:
    print(f"Model already converted: {model_path}")

### Select Inference Device

[back to top ⬆️](#Table-of-contents:)

In [ ]:
from notebook_utils import device_widget

device = device_widget("CPU", exclude=["NPU"])

device

### Load Model with OpenVINO

[back to top ⬆️](#Table-of-contents:)

OpenVINO models can be run locally through the `HuggingFacePipeline` class in LangChain. To deploy a model with OpenVINO, you can specify the `backend="openvino"` parameter to trigger OpenVINO as backend inference framework. For [more information](https://python.langchain.com/docs/integrations/llms/openvino/).

In [ ]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

import openvino.properties as props
import openvino.properties.hint as hints
import openvino.properties.streams as streams

ov_config = {hints.performance_mode(): hints.PerformanceMode.LATENCY, streams.num(): "1", props.cache_dir(): ""}

ov_llm = HuggingFacePipeline.from_model_id(
    model_id=str(model_path),
    task="text-generation",
    backend="openvino",
    model_kwargs={
        "device": device.value,
        "ov_config": ov_config,
        "trust_remote_code": True,
    },
    pipeline_kwargs={"max_new_tokens": 2048},
)

chat_model = ChatHuggingFace(llm=ov_llm, verbose=True)

## Build the Knowledge Base

[back to top ⬆️](#Table-of-contents:)

We create a small knowledge base from a web page to demonstrate the RAG pipeline. The documents are split into chunks and embedded into a ChromaDB vector store.

### Load and Chunk Documents

[back to top ⬆️](#Table-of-contents:)

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Load a sample document about OpenVINO
loader = WebBaseLoader(
    web_paths=["https://docs.openvino.ai/latest/about-openvino.html"],
)
docs = loader.load()

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)
splits = text_splitter.split_documents(docs)

print(f"Loaded {len(splits)} document chunks")

### Create Embeddings and Vector Store

[back to top ⬆️](#Table-of-contents:)

We use `bge-small-en-v1.5` as the embedding model and store vectors in ChromaDB.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model_id = "BAAI/bge-small-en-v1.5"

embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_id,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="openvino-docs",
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"Vector store created with {len(splits)} documents")

## Define the RAG Agent Graph

[back to top ⬆️](#Table-of-contents:)

Now we define the core of the self-correcting RAG agent using LangGraph. The agent is a state machine with four nodes:

- **retrieve**: fetch documents from the vector store
- **grade_documents**: use the LLM to judge document relevance
- **generate**: produce an answer from relevant documents
- **rewrite_query**: reformulate the query if documents are irrelevant

The graph has a conditional edge after grading: if documents are relevant, proceed to generation; otherwise, rewrite the query and re-retrieve.

### Define Agent State

[back to top ⬆️](#Table-of-contents:)

In [ ]:
from typing import TypedDict
from langchain_core.documents import Document


class AgentState(TypedDict):
    """State of the self-correcting RAG agent."""

    question: str
    documents: list[Document]
    generation: str
    rewrite_count: int
    max_rewrites: int

### Retrieval Node

[back to top ⬆️](#Table-of-contents:)

In [ ]:
def retrieve(state: AgentState) -> AgentState:
    """Retrieve documents from the vector store."""
    question = state["question"]
    display_query = question[:80] + "..." if len(question) > 80 else question
    print(f"--- RETRIEVE (query: {display_query}) ---")
    documents = retriever.invoke(question)
    return {"documents": documents}

### Document Grader Node

[back to top ⬆️](#Table-of-contents:)

The grader uses the LLM to determine whether each retrieved document is relevant to the question. If at least one document is relevant, the agent proceeds to generation. Otherwise, it rewrites the query.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

GRADER_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a document relevance grader. Given a user question and a retrieved document, "
            "determine if the document contains information relevant to answering the question. "
            "Respond with only 'yes' or 'no'.",
        ),
        (
            "human",
            "Question: {question}\n\nDocument:\n{document}\n\nIs this document relevant? (yes/no)",
        ),
    ]
)

grader_chain = GRADER_PROMPT | chat_model | StrOutputParser()


def grade_documents(state: AgentState) -> AgentState:
    """Grade retrieved documents for relevance."""
    print("--- GRADE DOCUMENTS ---")
    question = state["question"]
    documents = state["documents"]

    relevant_docs = []
    for doc in documents:
        score = grader_chain.invoke(
            {"question": question, "document": doc.page_content[:300]}
        )
        grade = "yes" if "yes" in score.lower() else "no"
        if grade == "yes":
            print(f"  + Relevant: {doc.page_content[:60]}")
            relevant_docs.append(doc)
        else:
            print(f"  - Not relevant: {doc.page_content[:60]}")

    return {"documents": relevant_docs}

### Answer Generator Node

[back to top ⬆️](#Table-of-contents:)

In [ ]:
GENERATE_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an assistant for question-answering tasks. Use the following pieces of "
            "retrieved context to answer the question. If you don't know the answer, just say "
            "that you don't know. Use three sentences maximum and keep the answer concise.",
        ),
        (
            "human",
            "Context:\n{context}\n\nQuestion: {question}",
        ),
    ]
)

generate_chain = GENERATE_PROMPT | chat_model | StrOutputParser()


def generate(state: AgentState) -> AgentState:
    """Generate an answer using relevant documents."""
    print("--- GENERATE ---")
    context = "\n\n".join(doc.page_content for doc in state["documents"])
    generation = generate_chain.invoke(
        {"context": context, "question": state["question"]}
    )
    return {"generation": generation}

### Query Rewriter Node

[back to top ⬆️](#Table-of-contents:)

In [ ]:
REWRITE_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a query rewriter. Given a user question that did not retrieve "
            "relevant documents, rewrite it to be more specific and likely to match "
            "relevant content. Return only the rewritten question.",
        ),
        (
            "human",
            "Original question: {question}\n\nRewritten question:",
        ),
    ]
)

rewrite_chain = REWRITE_PROMPT | chat_model | StrOutputParser()


def rewrite_query(state: AgentState) -> AgentState:
    """Rewrite the query to improve retrieval."""
    print("--- REWRITE QUERY ---")
    new_question = rewrite_chain.invoke({"question": state["question"]})
    # Take only the first line to avoid prompt leakage
    new_question = new_question.strip().split("\n")[0]
    print(f"  Rewritten: {new_question}")
    rewrite_count = state.get("rewrite_count", 0) + 1
    return {"question": new_question, "rewrite_count": rewrite_count}

### Routing Logic

[back to top ⬆️](#Table-of-contents:)

After grading, the agent decides whether to generate an answer or rewrite the query. If no relevant documents were found and we have not exceeded the maximum number of rewrites, the query is rewritten.

In [ ]:
def decide_next_step(state: AgentState) -> str:
    """Route to generation or query rewriting based on grading results."""
    if state["documents"]:
        print("--- DECISION: Relevant documents found -> GENERATE ---")
        return "generate"

    max_rewrites = state.get("max_rewrites", 2)
    rewrite_count = state.get("rewrite_count", 0)

    if rewrite_count >= max_rewrites:
        print(f"--- DECISION: Max rewrites ({max_rewrites}) reached -> GENERATE with available context ---")
        return "generate"

    print("--- DECISION: No relevant documents -> REWRITE QUERY ---")
    return "rewrite"

### Assemble the Graph

[back to top ⬆️](#Table-of-contents:)

In [ ]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("generate", generate)
workflow.add_node("rewrite_query", rewrite_query)

# Set entry point
workflow.set_entry_point("retrieve")

# Add edges
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    decide_next_step,
    {
        "generate": "generate",
        "rewrite": "rewrite_query",
    },
)
workflow.add_edge("rewrite_query", "retrieve")
workflow.add_edge("generate", END)

# Compile
app = workflow.compile()

print("Agent graph compiled successfully!")

In [ ]:
# Visualize the agent graph (optional, requires pygraphviz or mermaid)
try:
    from IPython.display import Image, display

    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    # Fall back to text representation if visualization deps are not available
    print("Graph nodes:", list(app.get_graph().nodes))
    print("Graph edges:", list(app.get_graph().edges))

## Run the Agent

[back to top ⬆️](#Table-of-contents:)

Let's test the self-correcting RAG agent with a question about OpenVINO.

In [ ]:
# Test with a relevant question
result = app.invoke(
    {
        "question": "What hardware does OpenVINO support for inference?",
        "documents": [],
        "generation": "",
        "rewrite_count": 0,
        "max_rewrites": 2,
    }
)

print("\n" + "=" * 60)
print("FINAL ANSWER:")
print("=" * 60)
print(result["generation"])

In [ ]:
# Test with a vague question that may trigger query rewriting
result = app.invoke(
    {
        "question": "How do I speed up my model?",
        "documents": [],
        "generation": "",
        "rewrite_count": 0,
        "max_rewrites": 2,
    }
)

print("\n" + "=" * 60)
print("FINAL ANSWER:")
print("=" * 60)
print(result["generation"])

## Interactive Demo with Gradio

[back to top ⬆️](#Table-of-contents:)

Launch an interactive interface to chat with the self-correcting RAG agent. The interface shows each step the agent takes: retrieval, grading, optional query rewriting, and final generation.

In [ ]:
import gradio as gr


def run_agent(question: str) -> str:
    """Run the self-correcting RAG agent and return a formatted response."""
    import io
    import contextlib

    # Capture print output to show agent reasoning
    log_buffer = io.StringIO()
    with contextlib.redirect_stdout(log_buffer):
        result = app.invoke(
            {
                "question": question,
                "documents": [],
                "generation": "",
                "rewrite_count": 0,
                "max_rewrites": 2,
            }
        )

    reasoning = log_buffer.getvalue()
    answer = result.get("generation", "No answer generated.")

    output = f"**Agent Reasoning:**\n```\n{reasoning}```\n\n**Answer:**\n{answer}"
    return output


demo = gr.Interface(
    fn=run_agent,
    inputs=gr.Textbox(
        label="Question",
        placeholder="Ask a question about OpenVINO...",
        lines=2,
    ),
    outputs=gr.Markdown(label="Response"),
    title="Self-Correcting RAG Agent with OpenVINO",
    description="Ask questions about OpenVINO. The agent retrieves documents, grades their relevance, and rewrites the query if needed.",
    examples=[
        ["What hardware does OpenVINO support?"],
        ["How do I optimize a model for Intel GPUs?"],
        ["What is the difference between FP16 and INT8 quantization?"],
    ],
    allow_flagging="never",
)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(share=True, debug=True)
# If you are launching remotely, specify server_name and server_port
# EXAMPLE: `demo.launch(server_name='your server name', server_port='server port in int')`
# To learn more please refer to the Gradio docs: https://gradio.app/docs/

In [ ]:
# # Cleanup: uncomment to remove downloaded model files
# import shutil
# model_dir = Path(llm_model_id.value.split("/")[-1])
# if model_dir.exists():
#     shutil.rmtree(model_dir)
#     print(f"Removed {model_dir}")